In [ ]:
include("../src/PhasorNetworks.jl")
using .PhasorNetworks, Plots, DifferentialEquations

In [ ]:
using Lux, LuxCUDA

In [ ]:
gdev = gpu_device()
cdev = cpu_device()

In [ ]:
n_x = 21
n_y = 21
n_vsa = 1

In [ ]:
repeats = 6
tspan = (0.0, repeats*1.0)

In [ ]:
function bundling_test(spk_args::SpikingArgs, device="cpu")
    tbase = collect(0.0:0.01:tspan[2])
    phases = collect([[x, y] for x in range(-1.0, 1.0, n_x), y in range(-1.0, 1.0, n_y)]) |> stack
    phases = reshape(phases, (1,2,:))
    
    b = v_bundle(phases, dims=2)
    st = phase_to_train(phases, spk_args=spk_args, repeats=6)

    if device == "gpu"
        st = SpikeTrainGPU(st)
    end

    #check potential encodings
    b2_sol = v_bundle(st, dims=2, spk_args=spk_args, tspan=tspan, return_solution=true)
    b2_phase = solution_to_phase(b2_sol, tbase, spk_args=spk_args, offset=0.0)

    if device == "gpu"
        b2_phase = b2_phase |> cdev
    end

    b2_phase_error = vec(b2_phase[1,1,:,end]) .- vec(b)
    
    return b2_sol, b2_phase_error
end

In [ ]:
spk_args = SpikingArgs(solver = Heun(),
                    solver_args = Dict(:adaptive => false, 
                                    :dt => 0.01),
                                    threshold = 0.001)

In [ ]:
cpu_sol, cpu_pe = bundling_test(spk_args, "cpu")

In [ ]:
histogram(arc_error.(cpu_pe))

In [ ]:
gpu_sol, gpu_pe = bundling_test(spk_args, "gpu")

In [ ]:
histogram(arc_error.(gpu_pe))

In [ ]:
gpu_pe .- cpu_pe |> unique

In [ ]:
maximum(gpu_pe .- cpu_pe)

In [ ]:
import .PhasorNetworks: bias_current, gaussian_kernel, is_active

In [ ]:
spk_args.t_period

In [ ]:
biases = collect(-1.0:0.01:1.0);

In [ ]:
tms = phase_to_time(biases, spk_args.t_period, 0.0)

In [ ]:
mag = ones(size(biases));

In [ ]:
bc = bias_current(biases, mag, 0.5, 0.0, spk_args);

In [ ]:
bc2 = bias_current(biases, mag, 0.25, 0.0, spk_args);

In [ ]:
bc3 = bias_current(biases, mag, 0.75, 0.0, spk_args);

In [ ]:
plot(bc)
plot!(bc2)
plot!(bc3)